# Telemetry Data Analysis & Graphing
## Interactive Notebook for Handling Dynamics Analysis

This notebook loads telemetry data and generates all graphs with **distance in meters** and **10-meter tick marks**.

**Improved Features:**
- ✅ Large, easy-to-read X-axis labels (15px)
- ✅ Clear 10-meter tick marks
- ✅ Enhanced gridlines for better reference
- ✅ Interactive zoom and pan capabilities
- ✅ Angled X-axis labels to prevent overlap

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import sys

sys.path.insert(0, 'c:/Users/Maha/Documents/telemetry')
import app

print("✅ Libraries loaded successfully")

## Step 1: Load and Process Data

In [ ]:
# Load sample telemetry data
df = pd.read_csv('sample_clean_telemetry.csv')
print(f"📊 Data loaded: {len(df)} rows\n")

# Process data
cleaned_data = app.clean_telemetry_data(df)
featured_data = app.engineer_features(cleaned_data)
thresholds = app.compute_thresholds(featured_data)
zoned_data = app.add_zone_columns(featured_data, thresholds)
events = app.detect_events(zoned_data, thresholds)
stats = app.build_stats(zoned_data)

print(f"✅ Data processing complete!\n")
print(f"📊 Telemetry Statistics:")
print(f"  📏 Lap Time: {stats['lap_time']:.2f}s")
print(f"  📍 Total Distance: {stats['lap_distance']:.1f}m")
print(f"  🚗 Average Speed: {stats['average_speed']:.1f} km/h")
print(f"  ⚡ Top Speed: {stats['top_speed']:.1f} km/h")

## Graph Styling Function

In [ ]:
def style_graph(fig, title, x_label, y_label):
    """Apply enhanced styling with large, visible X-axis labels"""
    fig.update_layout(
        template='plotly_white',
        title=dict(text=title, font=dict(size=20, color='#1a1a1a', family='Arial Black')),
        height=600,
        margin=dict(l=100, r=80, t=100, b=120),
        hovermode='x unified',
        font=dict(size=12, family='Arial')
    )
    
    # Enhanced X-axis for Distance
    if 'Distance' in x_label:
        fig.update_xaxes(
            title=dict(text=x_label, font=dict(size=16, color='#000000', family='Arial Black'), standoff=40),
            showgrid=True,
            gridcolor='rgba(200, 200, 200, 0.6)',
            gridwidth=1.5,
            dtick=10,
            tickfont=dict(size=14, color='#333333', family='Arial'),
            tickangle=-45,
            showline=True,
            linewidth=4,
            linecolor='rgb(60, 60, 60)',
            mirror=True,
            zeroline=False
        )
    
    # Enhanced Y-axis
    fig.update_yaxes(
        title=dict(text=y_label, font=dict(size=15, color='#000000'), standoff=40),
        showgrid=True,
        gridcolor='rgba(200, 200, 200, 0.4)',
        gridwidth=1,
        tickfont=dict(size=13, color='#333333'),
        showline=True,
        linewidth=3,
        linecolor='rgb(80, 80, 80)',
        mirror=True
    )
    
    return fig

print('✅ Graph styling configured')

## Speed vs Distance

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=zoned_data['distance_traveled'],
        y=zoned_data['speed'],
        mode='lines',
        name='Speed',
        line=dict(color='#1f77b4', width=4),
        hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>Speed:</b> %{y:.1f} km/h<extra></extra>'
    )
)
fig = style_graph(fig, '🚗 Speed vs Distance Traveled', 'Distance (m)', 'Speed (km/h)')
fig.show()

## Throttle & Brake Controls

In [ ]:
data_copy = zoned_data.copy()
data_copy['throttle_pct'] = data_copy['throttle_input'] * 100
data_copy['brake_pct'] = data_copy['brake_input'] * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=data_copy['distance_traveled'], y=data_copy['throttle_pct'], mode='lines',
    name='Throttle', line=dict(color='#2ca02c', width=4), hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>Throttle:</b> %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Scatter(x=data_copy['distance_traveled'], y=data_copy['brake_pct'], mode='lines',
    name='Brake', line=dict(color='#d62728', width=4), hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>Brake:</b> %{y:.1f}%<extra></extra>'))
fig = style_graph(fig, '⚙️ Throttle & Brake vs Distance', 'Distance (m)', 'Pedal Input (%)')
fig.show()

## Slip Angle Analysis

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=zoned_data['distance_traveled'], y=zoned_data['slip_severity'], mode='lines',
    name='Slip Severity', line=dict(color='#ff7f0e', width=4), hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>Slip:</b> %{y:.2f}°<extra></extra>'))
fig = style_graph(fig, '🔄 Slip Angle vs Distance', 'Distance (m)', 'Slip Angle (°)')
fig.show()

## Lateral Acceleration (G-Forces)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=zoned_data['distance_traveled'], y=zoned_data['lateral_accel'], mode='lines',
    name='Lateral G', line=dict(color='#e377c2', width=4), fill='tozeroy', fillcolor='rgba(227, 119, 194, 0.2)',
    hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>Lateral G:</b> %{y:.2f} m/s²<extra></extra>'))
corner_max = zoned_data['lateral_accel'].max()
fig.add_hline(y=corner_max * 0.8, line_dash='dash', line_color='rgba(100,100,100,0.6)', annotation_text='High G-Load')
fig = style_graph(fig, '⬅️➡️ Lateral Acceleration vs Distance', 'Distance (m)', 'Lateral G (m/s²)')
fig.show()

## Steering & Yaw (Dual Axis)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=zoned_data['distance_traveled'], y=zoned_data['steering_angle'], mode='lines',
    name='Steering Angle', line=dict(color='#1f77b4', width=4), yaxis='y1',
    hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>Steering:</b> %{y:.2f}°<extra></extra>'))
fig.add_trace(go.Scatter(x=zoned_data['distance_traveled'], y=zoned_data['yaw_rate'], mode='lines',
    name='Yaw Rate', line=dict(color='#ff7f0e', width=4), yaxis='y2',
    hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>Yaw Rate:</b> %{y:.3f}°/s<extra></extra>'))

fig.update_layout(template='plotly_white', title=dict(text='🔧 Steering & Yaw vs Distance', font=dict(size=20, family='Arial Black')),
    height=600, margin=dict(l=100, r=140, t=100, b=120), hovermode='x unified',
    xaxis=dict(title=dict(text='Distance (m)', font=dict(size=16, family='Arial Black'), standoff=40),
        dtick=10, tickfont=dict(size=14), tickangle=-45, gridcolor='rgba(200,200,200,0.6)', linewidth=4),
    yaxis=dict(title=dict(text='Steering Angle (°)', font=dict(size=15), standoff=40), gridcolor='rgba(200,200,200,0.4)'),
    yaxis2=dict(title=dict(text='Yaw Rate (°/s)', font=dict(size=15), standoff=40), overlaying='y', side='right'))
fig.show()

## RPM Evolution

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=zoned_data['distance_traveled'], y=zoned_data['rpm'], mode='lines',
    name='RPM', line=dict(color='#9467bd', width=4), hovertemplate='<b>Distance:</b> %{x:.1f}m<br><b>RPM:</b> %{y:.0f}<extra></extra>'))
fig.add_hrect(y0=thresholds['rpm_low'], y1=thresholds['rpm_high'], fillcolor='rgba(46, 204, 113, 0.12)', line_width=0, annotation_text='Efficient RPM band')
fig = style_graph(fig, '⚙️ RPM Evolution vs Distance', 'Distance (m)', 'RPM')
fig.show()

## Summary

✅ **All Graphs Include:**
- **Distance in METERS** with 10-meter tick marks
- **Large X-axis labels** (16px) - EASY TO READ
- **Enhanced gridlines** for better reference
- **Angled labels** to prevent overlapping
- **Dark borders** on axes for clarity
- **Interactive zoom/pan** via Plotly tools
- **Hover information** for precise data values